# Agent Systems - Project PoC

## AI agent document assistant

### Environment setup

In [ ]:
!pip install -qU langchain langchain-community langchain-huggingface chromadb pypdf docx2txt sentence-transformers langchain-ollama langchain-openai langgraph

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.3/114.3 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 65.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 71.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.8/343.8 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.9/588.9 kB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.3/235.3 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 92.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5

Install ollama and pull tool calling supported model \(llama3.1\).

This mimics the local environment setup.

In [ ]:
!sudo apt-get install zstd
!curl -fsSL https://ollama.com/install.sh | sh

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 51 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (7,790 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 122363 files and directories currently

In [ ]:
!nohup ollama serve > ollama.log 2>&1 &

!sleep 3

!ollama pull llama3.1

# **Planned** Agentic RAG System Architecture and File Structure

This project strictly adheres to the Single Responsibility Principle (SRP) and Separation of Concerns to ensure scalability from a single-agent orchestrator to a LangGraph-driven multi-agent system.

## Directory Structure **\(Planned\)**

```text
rag_project/
│
├── core/                           # Data & Core Capabilities
│   ├── __init__.py
│   ├── vector_store.py             # VectorStoreManager (Wraps ChromaDB)
│   ├── repository.py               # SessionRepository (Reads/Writes State JSONs)
│   ├── ingestor.py                 # DocumentIngestor (ETL: Load -> Chunk -> Embed)
│   └── tools/                      # ToolRegistry
│       ├── __init__.py
│       ├── search_tool.py          # Semantic Search (RAG)
│       ├── summary_tool.py         # Document Summarization
│       └── metadata_tool.py        # Structure/TOC Navigation
│
├── orchestration/                  # Orchestration & State (The Brain)
│   ├── __init__.py
│   ├── state.py                    # Session State Models (Pydantic/Dataclasses)
│   ├── manager.py                  # SessionManager (Handles state context switching)
│   └── agent.py                    # AgentOrchestrator
│
├── api/                            # The Interface
│   ├── __init__.py
│   └── assistant.py                # Assistant Facade (Exposes clean API to UI)
│
├── ui/                             # THE FRONTEND
│   └── app.py                      # TBD: Streamlit / FastAPI / Chainlit UI
│
├── data/                           # PERSISTENCE
│   ├── sessions/                   # JSON files for saved session states
│   └── chroma_db/                  # Local embedded Chroma vectors

### Architectural Flow
Facade Entry: The ui/app.py only ever imports api/assistant.py.

#### Context Loading:
Assistant fetches state via SessionManager, which uses SessionRepository.

#### Action Routing:

**Uploads:**

Routed to DocumentIngestor -> Updates VectorStoreManager.

**Chats:**

Routed to AgentOrchestrator -> Uses tools/ -> Returns response.


# POC - Implementation

In [ ]:
import os
import hashlib
from typing import Dict, List, Any
from langchain_community.document_loaders import (
    PyPDFLoader,
    Docx2txtLoader,
    TextLoader,
    UnstructuredMarkdownLoader
)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

class SessionManager:
    """
    Combines Document Ingestion and Session State management.
    Tracks registry, chat history, and vector storage.
    """

    def __init__(self, persist_directory: str = "./chroma_session_db"):
        self.persist_directory = persist_directory

        # Session State Data
        self.state = {
            "registry": {},      # {source_id: {metadata}}
            "chat_history": [],  # List of message objects
            "tool_logs": []      # Sequence of executed actions
        }

        # Embeddings & Vector Store
        self.embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
        self.vector_store = Chroma(
            collection_name="rag_session",
            embedding_function=self.embeddings,
            persist_directory=self.persist_directory
        )

        # Standard Chunking Strategy
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000,
            chunk_overlap=200
        )

    def _get_loader(self, file_path: str):
        """Returns the appropriate loader or a default fallback."""
        ext = os.path.splitext(file_path)[1].lower()

        if ext == '.pdf':
            return PyPDFLoader(file_path)
        elif ext == '.docx':
            return Docx2txtLoader(file_path)
        elif ext == '.md':
            return UnstructuredMarkdownLoader(file_path)
        elif ext == '.txt':
            return TextLoader(file_path)
        else:
            # DEFAULT FALLBACK: Try loading as text if unknown
            print(f"⚠️ Unknown extension '{ext}'. Attempting default TextLoader.")
            return TextLoader(file_path)

    def add_document(self, file_path: str):
        file_name = os.path.basename(file_path)
        source_id = hashlib.md5(file_name.encode()).hexdigest()

        if source_id in self.state["registry"]:
            return source_id

        loader = self._get_loader(file_path)
        docs = loader.load()

        # Metadata Injection
        for doc in docs:
            doc.metadata.update({
                "source_id": source_id,
                "file_name": file_name,
                "type": os.path.splitext(file_name)[1][1:] or "txt"
            })

        chunks = self.text_splitter.split_documents(docs)
        self.vector_store.add_documents(chunks)

        # Update Session Registry
        self.state["registry"][source_id] = {
            "file_name": file_name,
            "chunks": len(chunks),
            "type": doc.metadata["type"]
        }
        return source_id

    def add_chat_message(self, role: str, content: str):
        """Stores message history for multi-turn awareness."""
        self.state["chat_history"].append({"role": role, "content": content})

    def get_summary(self):
        """Displays ingestion and session status."""
        print("\n--- 📊 SESSION STATUS ---")
        print(f"Files Ingested: {len(self.state['registry'])}")
        for sid, meta in self.state["registry"].items():
            print(f"  • {meta['file_name']} ({meta['chunks']} chunks)")
        print(f"Chat History: {len(self.state['chat_history'])} messages")

/tmp/ipykernel_2743/2101197510.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (


### Storage

In [ ]:
# core/vector_store.py
from typing import List
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

class VectorStoreManager:
    """
    Manages the lifecycle of the embedding model and the ChromaDB connection.
    Follows Lazy Initialization to save memory.
    """
    def __init__(self, persist_directory: str = "./data/chroma_db"):
        self.persist_directory = persist_directory
        self._embeddings = None
        self._db = None

    @property
    def embeddings(self):
        """Lazy load the heavy HuggingFace model only when first requested."""
        if self._embeddings is None:
            print("Loading HuggingFace Embeddings into memory...")
            self._embeddings = HuggingFaceEmbeddings(
                model_name="all-MiniLM-L6-v2",
                model_kwargs={'device': 'cuda'} # Colab T4 GPU
            )
        return self._embeddings

    @property
    def db(self):
        """Lazy load the ChromaDB collection."""
        if self._db is None:
            self._db = Chroma(
                collection_name="rag_global",
                embedding_function=self.embeddings,
                persist_directory=self.persist_directory
            )
        return self._db

    def add_documents(self, documents: List[Document]):
        """Saves chunks to the database."""
        self.db.add_documents(documents)

    def delete_by_source(self, source_id: str):
        """Removes a specific document from the database."""
        if self._db:
            self._db._collection.delete(where={"source_id": source_id})

### Ingest

In [ ]:
# core/ingestor.py
import os
import hashlib
from typing import Dict, Any
from langchain_community.document_loaders import (
    PyPDFLoader,
    Docx2txtLoader,
    TextLoader,
    UnstructuredMarkdownLoader
)
from langchain_text_splitters import RecursiveCharacterTextSplitter
# from core.vector_store import VectorStoreManager

class DocumentIngestor:
    """
    Pure ETL Pipeline: Extracts text, transforms to chunks, and loads to Vector DB.
    Strictly stateless. Returns metadata 'receipts' to the caller.
    """
    def __init__(self, vector_store: VectorStoreManager):
        self.vector_store = vector_store

        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000,
            chunk_overlap=200
        )

    def _get_loader(self, file_path: str):
        """Factory method for document loaders."""
        ext = os.path.splitext(file_path)[1].lower()
        if ext == '.pdf':
            return PyPDFLoader(file_path)
        elif ext == '.docx':
            return Docx2txtLoader(file_path)
        elif ext == '.md':
            return UnstructuredMarkdownLoader(file_path)
        else:
            return TextLoader(file_path)

    def process(self, file_path: str, session_id: str) -> Dict[str, Any]:
        """
        Executes the ingestion pipeline and returns a metadata receipt.
        """
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"File not found: {file_path}")

        file_name = os.path.basename(file_path)
        source_id = hashlib.md5(f"{session_id}_{file_name}".encode()).hexdigest()

        # Extract
        print(f"[{session_id}] Extracting data from {file_name}...")
        loader = self._get_loader(file_path)
        docs = loader.load()

        file_type = os.path.splitext(file_name)[1][1:] or "txt"

        # Transform (Metadata Injection & Chunking)
        for doc in docs:
            doc.metadata.update({
                "source_id": source_id,
                "session_id": session_id,
                "file_name": file_name,
                "file_type": file_type
            })
            if "page" not in doc.metadata:
                doc.metadata["page"] = 0

        chunks = self.text_splitter.split_documents(docs)

        # Persist in Vector Storage
        print(f"[{session_id}] Generating embeddings for {len(chunks)} chunks...")
        self.vector_store.add_documents(chunks)

        # Return Receipt (For the SessionManager to save in JSON state)
        return {
            "source_id": source_id,
            "file_name": file_name,
            "file_type": file_type,
            "total_chunks": len(chunks),
            "status": "ingested"
        }

### Session

In [ ]:
# orchestration/state.py
from dataclasses import dataclass, field
from typing import List, Dict, Any

@dataclass
class SessionState:
    session_id: str
    # Registry maps source_id -> Document Metadata Receipt
    registry: Dict[str, Dict[str, Any]] = field(default_factory=dict)
    # Chat history is a list of standardized message dictionaries
    chat_history: List[Dict[str, str]] = field(default_factory=list)
    # Tool logs for debugging the ReAct loop
    tool_logs: List[Dict[str, Any]] = field(default_factory=list)
    # Store runtime filters
    active_filters: List[str] = field(default_factory=list)

    def to_dict(self) -> Dict[str, Any]:
        return {
            "session_id": self.session_id,
            "registry": self.registry,
            "chat_history": self.chat_history,
            "tool_logs": self.tool_logs
        }


In [ ]:
# core/repository.py
import os
import json
# from orchestration.state import SessionState

class SessionRepository:
    """Handles the persistence of the SessionState to the filesystem."""

    def __init__(self, storage_dir: str = "./data/sessions"):
        self.storage_dir = storage_dir
        os.makedirs(self.storage_dir, exist_ok=True)

    def _get_filepath(self, session_id: str) -> str:
        return os.path.join(self.storage_dir, f"{session_id}.json")

    def save(self, state: SessionState):
        """Writes the state to a JSON file."""
        filepath = self._get_filepath(state.session_id)
        with open(filepath, "w", encoding="utf-8") as f:
            json.dump(state.to_dict(), f, indent=4)

    def load(self, session_id: str) -> SessionState:
        """Loads a state from disk, or creates a new one if it doesn't exist."""
        filepath = self._get_filepath(session_id)
        if os.path.exists(filepath):
            with open(filepath, "r", encoding="utf-8") as f:
                data = json.load(f)
                return SessionState(
                    session_id=data["session_id"],
                    registry=data.get("registry", {}),
                    chat_history=data.get("chat_history", []),
                    tool_logs=data.get("tool_logs", [])
                )
        # Return a fresh state if no file exists
        return SessionState(session_id=session_id)

    def set_active_filters(self, source_ids: List[str] = None):
        """Temporarily scopes the session to specific documents for the next agent run."""
        if not self.active_state:
            raise ValueError("No active session loaded.")

        # If None or empty, we clear the filters (search all)
        self.active_state.active_filters = source_ids or []

In [ ]:
# orchestration/manager.py
# from core.ingestor import DocumentIngestor
# from core.repository import SessionRepository
# from orchestration.state import SessionState

class SessionManager:
    """
    The central orchestrator for state. It routes document tasks to the Ingestor
    and maintains the conversational context.
    """

    def __init__(self, ingestor: DocumentIngestor, repository: SessionRepository):
        self.ingestor = ingestor
        self.repository = repository
        self.active_state: SessionState = None

    def load_session(self, session_id: str):
        """Loads a session into active memory."""
        print(f"Loading context for session: {session_id}")
        self.active_state = self.repository.load(session_id)

    def save_session(self):
        """Persists the current active state to the repository."""
        if self.active_state:
            self.repository.save(self.active_state)

    def add_document(self, file_path: str) -> str:
        """
        Delegates processing to the stateless Ingestor, then updates
        the session's state with the resulting receipt.
        """
        if not self.active_state:
            raise ValueError("No active session loaded. Call load_session() first.")

        session_id = self.active_state.session_id

        # Ingest
        receipt = self.ingestor.process(file_path, session_id)

        # Update State
        source_id = receipt["source_id"]
        self.active_state.registry[source_id] = receipt

        # Save progress
        self.save_session()
        print(f"Document registered to session {session_id}. Receipt: {source_id}")
        return source_id

    def add_message(self, role: str, content: str):
        """Appends a message to the multi-turn chat history."""
        if not self.active_state:
            raise ValueError("No active session loaded.")

        self.active_state.chat_history.append({"role": role, "content": content})
        self.save_session()

    def get_context(self) -> SessionState:
        """Returns the active state for the AgentOrchestrator to read."""
        return self.active_state

    def set_active_filters(self, source_ids: List[str] = None):
        """Temporarily scopes the session to specific documents for the next agent run."""
        if not self.active_state:
            raise ValueError("No active session loaded.")

        # If None or empty clear the filters
        self.active_state.active_filters = source_ids or []

### Tools

In [ ]:
# core/tools/registry.py
from typing import List, Type
from langchain_core.tools import BaseTool, tool
# from core.vector_store import VectorStoreManager
# from orchestration.manager import SessionManager

class ToolRegistry:
    """
    Acts as a factory to build and return LangChain tools.
    Injects necessary managers so tools can access the data layer and session state.
    """
    def __init__(self, vector_store: VectorStoreManager, session_manager: SessionManager):
        self.vector_store = vector_store
        self.session_manager = session_manager

    def get_tools(self) -> List[BaseTool]:
        """Returns the suite of specialized tools for the General Agent."""

        # Semantic Search Tool (RAG)
        @tool
        def semantic_search(query: str) -> str:
            """
            Useful for searching the contents of the uploaded documents.
            Converts queries into vector embeddings to find the most relevant chunks.
            """
            state = self.session_manager.get_context()
            if not state or not state.registry:
                return "Error: No active documents in the current session."

            # Always lock search to the current session
            chroma_filter = {"session_id": state.session_id}

            # Apply UI Filters if they exist
            if hasattr(state, 'active_filters') and state.active_filters:
                # ChromaDB requires the $and operator to combine conditions
                chroma_filter = {
                    "$and": [
                        {"session_id": state.session_id},
                        {"source_id": {"$in": state.active_filters}}
                    ]
                }

            results = self.vector_store.db.similarity_search(
                query,
                k=4,
                filter=chroma_filter
            )

            if not results:
                return "No relevant information found in the active documents."

            formatted_results = "\n\n".join(
                [f"Source: {res.metadata.get('file_name')} (Page {res.metadata.get('page')}):\n{res.page_content}"
                 for res in results]
            )
            return formatted_results

        # Metadata/Structure Tool
        @tool
        def get_document_metadata() -> str:
            """
            Extracts the document's structure and metadata. Use this to find out
            what files are available, their names, and how large they are.
            """
            state = self.session_manager.get_context()
            if not state or not state.registry:
                return "No documents have been ingested yet."

            inventory = []
            for doc_id, meta in state.registry.items():
                inventory.append(
                    f"- File: {meta.get('file_name')} | Type: {meta.get('file_type')} | Chunks: {meta.get('total_chunks')}"
                )

            return "Active Documents:\n" + "\n".join(inventory)

        # Summarization Tool
        @tool
        def summarize_document(file_name: str) -> str:
            """
            Condenses specific files when a high-level overview is requested.
            Provide the exact file_name from the metadata tool.
            """
            state = self.session_manager.get_context()
            # Find the source_id for the given filename
            target_id = None
            for doc_id, meta in state.registry.items():
                if meta.get('file_name').lower() == file_name.lower():
                    target_id = doc_id
                    break

            if not target_id:
                return f"Error: File '{file_name}' not found in current session."

            # Fetch chunks
            results = self.vector_store.db.similarity_search(
                "introduction summary overview",
                k=5,
                filter={"source_id": target_id}
            )

            content = "\n".join([res.page_content for res in results])
            return f"Raw content extracted for summarization:\n{content}\n\n(Agent: Please synthesize this into a summary)."

        return [semantic_search, get_document_metadata, summarize_document]

### Agent

In [ ]:
# orchestration/agent.py
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage, AIMessage
from langchain_ollama import ChatOllama
from langchain_openai import ChatOpenAI

# from orchestration.manager import SessionManager
# from core.tools.registry import ToolRegistry

class AgentOrchestrator:
    """
    The General Orchestrator Agent (Phase 1).
    Evaluates user intent, executes actions, and synthesizes responses.
    """
    def __init__(self, session_manager: SessionManager,
                 tool_registry: ToolRegistry, use_local: bool = True,
                 temperature: float = 0.0, model: str = None):
        self.session_manager = session_manager
        self.tools = tool_registry.get_tools()
        self.temperature = temperature

        # Initialize LLM
        if use_local:
            print("Initializing Local LLM via Ollama...")
            self.llm = ChatOllama(model="llama3.1" if model is None else model, temperature=self.temperature)
        else:
            print("Initializing Cloud Fallback (OpenAI)...")
            self.llm = ChatOpenAI(model="gpt-4o-mini" if model is None else model, temperature=self.temperature)

        self.agent_executor = self._build_agent()

    def _build_agent(self):
        """Constructs the LangGraph v1 Agent Loop."""

        system_prompt = """You are a specialized General Orchestrator Agent.
        Your primary goal is to interact with uploaded document data.

        CRITICAL INSTRUCTIONS:
        1. If you need to use a tool, use the provided tool-calling mechanism. DO NOT output the raw JSON of the tool call to the user.
        2. Once you receive the 'Observation' from a tool, you MUST synthesize a natural language response.
        3. NEVER end your turn with a JSON object. ALWAYS provide a conversational, synthesized final answer based on the tool's output.
        4. Synthesize your responses strictly grounded in the source material. Do not hallucinate.
        """

        agent = create_agent(
            model=self.llm,
            tools=self.tools,
            system_prompt=system_prompt
        )

        return agent

    def invoke(self, user_input: str) -> str:
        """
        Takes the user input, injects the active session history,
        runs the agent graph loop, and saves the result.
        """
        state = self.session_manager.get_context()
        if not state:
            return "System Error: No active session loaded."

        # LangGraph state dictionary containing a list of standard message objects.
        messages = []
        for msg in state.chat_history:
            if msg["role"] == "user":
                messages.append(HumanMessage(content=msg["content"]))
            elif msg["role"] == "assistant":
                messages.append(AIMessage(content=msg["content"]))

        # Append the user input
        messages.append(HumanMessage(content=user_input))

        # Agent ReAct loop
        print(f"\n--- AGENT THINKING ---")

        response = self.agent_executor.invoke({
            "messages": messages
        })

        final_message = response["messages"][-1]
        output = final_message.content if final_message.content else "I could not generate a response."

        # Update Session State
        self.session_manager.add_message("user", user_input)
        self.session_manager.add_message("assistant", output)

        return output

### Assistant

In [ ]:
# api/assistant.py
from typing import Dict, Any
# from orchestration.manager import SessionManager
# from orchestration.agent import AgentOrchestrator

class Assistant:
    """
    The Facade Layer.
    Exposes a clean, simplified API for any UI (Streamlit, FastAPI, CLI) to consume.
    The UI should NEVER import LangChain or ChromaDB directly.
    """
    def __init__(self, session_manager: SessionManager, orchestrator: AgentOrchestrator):
        self._session_manager = session_manager
        self._orchestrator = orchestrator

    def set_session(self, session_id: str):
        """Initializes or switches the active conversational context."""
        self._session_manager.load_session(session_id)
        return f"Session '{session_id}' is now active."

    def upload_document(self, file_path: str) -> Dict[str, Any]:
        """
        Processes a document and attaches it to the active session.
        Returns a dictionary with the success status and metadata.
        """
        try:
            source_id = self._session_manager.add_document(file_path)
            return {"status": "success", "source_id": source_id}
        except Exception as e:
            return {"status": "error", "message": str(e)}

    def chat(self, message: str, allowed_sources: list[str] = None) -> str:
        """
        Sends a user message to the ReAct agent.
        Optionally filters the RAG search to specific source IDs.
        """
        if not self._session_manager.get_context():
            return "System Error: Please set a session before chatting."

        try:
            # Apply UI Checkbox Filters to the State
            self._session_manager.set_active_filters(allowed_sources)

            # Run the Agent
            response = self._orchestrator.invoke(message)
            return response
        except Exception as e:
            return f"Agent Error: {str(e)}"

    def get_status(self) -> Dict[str, Any]:
        """Returns a snapshot of the current session for the UI to render."""
        state = self._session_manager.get_context()
        if not state:
            return {"status": "No active session."}

        return {
            "session_id": state.session_id,
            "documents": list(state.registry.values()),
            "message_count": len(state.chat_history)
        }

### Build and test the assistant - ingest + query

In [ ]:
# from core.vector_store import VectorStoreManager
# from core.repository import SessionRepository
# from core.ingestor import DocumentIngestor
# from core.tools.registry import ToolRegistry
# from orchestration.manager import SessionManager
# from orchestration.agent import AgentOrchestrator
# from api.assistant import Assistant

def build_assistant(use_local_llm: bool = True) -> Assistant:
    """
    The Composition Root.
    Wires up the dependencies from Layer 1 (Data) up to Layer 4 (Facade).
    """
    print("Bootstrapping RAG Architecture...")

    vector_store = VectorStoreManager(persist_directory="./data/chroma_db")
    repository = SessionRepository(storage_dir="./data/sessions")

    ingestor = DocumentIngestor(vector_store=vector_store)

    session_manager = SessionManager(ingestor=ingestor, repository=repository)

    tool_registry = ToolRegistry(vector_store=vector_store, session_manager=session_manager)

    orchestrator = AgentOrchestrator(
        session_manager=session_manager,
        tool_registry=tool_registry,
        use_local=use_local_llm,
        temperature=0.0
    )

    assistant = Assistant(session_manager=session_manager, orchestrator=orchestrator)

    print("System ready.")
    return assistant

# ==========================================
# Example Usage (Simulating a UI interaction)
# ==========================================

assistant = build_assistant(use_local_llm=True)

assistant.set_session("agent_systems_poc_demo")

print(assistant.upload_document("/content/Agent Systems - Agent Definition and Interaction Modelling.pdf"))

print("\n--- UI Dashboard ---")
print(assistant.get_status())

print("\n--- Chat Interface ---")
user_msg = "What documents are available currently?"
print(f"User: {user_msg}")

agent_response = assistant.chat(user_msg)
print(f"Agent: {agent_response}")

Bootstrapping RAG Architecture...
Initializing Local LLM via Ollama...
System ready.
Loading context for session: agent_systems_poc_demo
[agent_systems_poc_demo] Extracting data from Agent Systems - Agent Definition and Interaction Modelling.pdf...
[agent_systems_poc_demo] Generating embeddings for 10 chunks...
Loading HuggingFace Embeddings into memory...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/tmp/ipykernel_2743/1136635696.py:32: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  self._db = Chroma(


Document registered to session agent_systems_poc_demo. Receipt: 912c3db0e8d1dfc01c53d9cb7ca80bbc
{'status': 'success', 'source_id': '912c3db0e8d1dfc01c53d9cb7ca80bbc'}

--- UI Dashboard ---
{'session_id': 'agent_systems_poc_demo', 'documents': [{'source_id': '912c3db0e8d1dfc01c53d9cb7ca80bbc', 'file_name': 'Agent Systems - Agent Definition and Interaction Modelling.pdf', 'file_type': 'pdf', 'total_chunks': 10, 'status': 'ingested'}], 'message_count': 0}

--- Chat Interface ---
User: What documents are available currently?

--- AGENT THINKING ---
Agent: The current available documents include 'Agent Systems - Agent Definition and Interaction Modelling.pdf', which is a PDF document with 10 chunks.


### UI

IPywidgets is used for the POC implementation

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

def create_colab_ui(assistant):
    """
    Builds an ipywidgets UI for the RAG Assistant in Google Colab.
    """
    # Initialize a default testing session
    assistant.set_session("agent_systems_poc_demo_ui")

    # ==========================================
    # DOCUMENT MANAGEMENT PANEL (Left Side)
    # ==========================================
    doc_header = widgets.HTML("<h3>Document Management</h3>")
    file_input = widgets.Text(
        description="File Path:",
        placeholder="/content/sample.pdf",
        layout=widgets.Layout(width='300px')
    )
    ingest_btn = widgets.Button(description="Ingest Document", button_style="info")
    ingest_output = widgets.Output()

    # Dynamic Checkbox area for sources
    sources_header = widgets.HTML("<b>Active Sources (Include in RAG):</b>")
    sources_box = widgets.VBox([])
    checkboxes_dict = {} # Maps source_id to the Checkbox widget

    def update_sources_ui():
        """Pulls the latest registry from the Assistant and updates checkboxes."""
        status = assistant.get_status()
        if "documents" in status and status["documents"]:
            cb_list = []
            for doc in status["documents"]:
                sid = doc.get("source_id", doc['file_name'])
                # Create a checkbox if it doesn't exist
                if sid not in checkboxes_dict:
                    checkboxes_dict[sid] = widgets.Checkbox(
                        value=True,
                        description=f"{doc['file_name']} ({doc.get('total_chunks', 0)} chunks)",
                        indent=False
                    )
                cb_list.append(checkboxes_dict[sid])
            sources_box.children = tuple(cb_list)
        else:
            sources_box.children = (widgets.HTML("<i>No documents ingested yet.</i>"),)

    def on_ingest_clicked(b):
        with ingest_output:
            clear_output()
            print(f"Uploading '{file_input.value}'...")
            result = assistant.upload_document(file_input.value)

            if result.get("status") == "success":
                print(f"✅ Success! (ID: {result.get('source_id')[:8]}...)")
                update_sources_ui()
                file_input.value = "" # Clear input
            else:
                print(f"❌ Error: {result.get('message')}")

    ingest_btn.on_click(on_ingest_clicked)

    # Assemble Left Panel
    left_panel = widgets.VBox([
        doc_header,
        widgets.HBox([file_input, ingest_btn]),
        ingest_output,
        widgets.HTML("<hr>"),
        sources_header,
        sources_box
    ], layout=widgets.Layout(width='40%', padding='10px', border='1px solid #ddd'))

    # ==========================================
    # CHAT INTERFACE PANEL (Right Side)
    # ==========================================
    chat_header = widgets.HTML("<h3>💬 Agent Chat</h3>")
    chat_output = widgets.Output(layout=widgets.Layout(
        height='350px',
        overflow='auto',
        border='1px solid #ccc',
        padding='10px',
        background_color='#f9f9f9'
    ))

    msg_input = widgets.Text(
        placeholder="Ask a question about your documents...",
        layout=widgets.Layout(width='80%')
    )
    send_btn = widgets.Button(description="Send", button_style="success")

    def on_send_clicked(b):
        user_text = msg_input.value.strip()
        if not user_text: return
        msg_input.value = ""

        # Determine which sources the user checked
        active_sources = [sid for sid, cb in checkboxes_dict.items() if cb.value]

        with chat_output:
            print(f"You: {user_text}")
            print(f"[System: Filtering to {len(active_sources)} selected sources...]")

            try:
                response = assistant.chat(user_text, allowed_sources=active_sources)

                print(f"\nAgent: {response}\n")
                print("-" * 50)
            except Exception as e:
                print(f"\n⚠️ UI Caught Error: {str(e)}\n")

    send_btn.on_click(on_send_clicked)
    msg_input.on_submit(lambda x: on_send_clicked(None)) # Allow 'Enter' to send

    # Assemble Right Panel
    right_panel = widgets.VBox([
        chat_header,
        chat_output,
        widgets.HBox([msg_input, send_btn], layout=widgets.Layout(margin='10px 0 0 0'))
    ], layout=widgets.Layout(width='55%', padding='10px'))

    # ==========================================
    # RENDER MAIN LAYOUT
    # ==========================================
    update_sources_ui() # Initial population
    main_layout = widgets.HBox([left_panel, right_panel], layout=widgets.Layout(width='100%', justify_content='space-between'))
    display(main_layout)

# Execute the UI
# Use previously initialized assistant

# Enter "/content/Agent Systems - Agent Definition and Interaction Modelling.pdf" as File Path for ingest
create_colab_ui(assistant)

Loading context for session: agent_systems_poc_demo_ui
